# 🏥 Medical Llama 3.1-8B — Lightning.AI Inference Server
**Loads base model + LoRA adapter | Strips reasoning tokens | FastAPI endpoint**

- No `<think>` tags exposed to users
- Professional doctor tone
- Chat memory injection (patient history)
- Safe medical disclaimer on every response

In [ ]:
# ─── 1. Install Dependencies ───────────────────────────────────────────────
!pip install -q transformers peft accelerate bitsandbytes huggingface_hub fastapi uvicorn pyngrok nest_asyncio

In [ ]:
# ─── 2. Imports ────────────────────────────────────────────────────────────
import os
import re
import torch
import uvicorn
import nest_asyncio
from threading import Thread
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TextIteratorStreamer,
)
from peft import PeftModel
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import Optional, List
from huggingface_hub import login

nest_asyncio.apply()
print(f'PyTorch   : {torch.__version__}')
print(f'GPU       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ─── 3. Configuration ──────────────────────────────────────────────────────
# HOW TO SET HF_TOKEN in Lightning.AI:
#   1. Open your Studio → click the ⚙ icon (top-right) → 'Secrets'
#   2. Add a secret named  HF_TOKEN  with your HuggingFace access token
#   3. Restart this notebook session so the env var is visible
#   OR: paste your token directly into the fallback string below (dev only).

_hf_token = os.environ.get('HF_TOKEN', '').strip()
if not _hf_token:
    raise EnvironmentError(
        "HF_TOKEN is not set or is empty.\n"
        "In Lightning.AI: Studio settings ⚙ → Secrets → add HF_TOKEN, "
        "then restart the session."
    )

INFERENCE_CFG = {
    'base_model'    : 'meta-llama/Llama-3.1-8B-Instruct',
    'lora_repo'     : 'YOUR_HF_USERNAME/medical-llama31-lora',  # your trained adapter repo
    'hf_token'      : _hf_token,

    # Generation defaults
    'max_new_tokens': 512,
    'temperature'   : 0.6,
    'top_p'         : 0.9,
    'repetition_penalty': 1.1,

    # Server
    'host'          : '0.0.0.0',
    'port'          : 8000,
    'use_ngrok'     : True,    # set False if running on Lightning.ai Studio with public URL
}

print(f'✅ Config loaded | HF_TOKEN: hf_...{_hf_token[-4:]} | model: {INFERENCE_CFG["base_model"]}')

In [ ]:
# ─── 4. Login to HuggingFace ───────────────────────────────────────────────
try:
    login(token=INFERENCE_CFG['hf_token'], add_to_git_credential=False)
    print('✅ HuggingFace authenticated')
except Exception as e:
    raise RuntimeError(
        f"HuggingFace login failed: {e}\n"
        "Check that your HF_TOKEN is a valid read/write token from "
        "https://huggingface.co/settings/tokens"
    ) from e

In [ ]:
# ─── 5. Load Tokenizer ─────────────────────────────────────────────────────
print('Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(
    INFERENCE_CFG['base_model'],
    token=INFERENCE_CFG['hf_token'],
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
print('✅ Tokenizer loaded')

In [ ]:
# ─── 6. Load Base Model (4-bit) ────────────────────────────────────────────
print('Loading base model with 4-bit quantization ...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    INFERENCE_CFG['base_model'],
    token=INFERENCE_CFG['hf_token'],
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
base_model.config.use_cache = True
print('✅ Base model loaded')

In [ ]:
# ─── 7. Attach LoRA Adapter ────────────────────────────────────────────────
print(f'Attaching LoRA adapter from {INFERENCE_CFG["lora_repo"]} ...')
model = PeftModel.from_pretrained(
    base_model,
    INFERENCE_CFG['lora_repo'],
    token=INFERENCE_CFG['hf_token'],
    is_trainable=False,
)
model.eval()
print('✅ LoRA adapter attached (inference mode)')

In [ ]:
# ─── 8. Response Cleaning — Strip ALL Reasoning Tokens ─────────────────────

# Patterns to strip from model output
_THINK_PATTERNS = [
    re.compile(r'<think>.*?</think>', re.DOTALL | re.IGNORECASE),
    re.compile(r'<thinking>.*?</thinking>', re.DOTALL | re.IGNORECASE),
    re.compile(r'<think>.*?$', re.DOTALL | re.IGNORECASE),       # unclosed tag
    re.compile(r'^.*?</think>', re.DOTALL | re.IGNORECASE),       # leading think block
    re.compile(r'Medical Reasoning Process:.*?(?=\n[A-Z]|$)', re.DOTALL),
    re.compile(r'Internal Reasoning:.*?(?=\n[A-Z]|$)', re.DOTALL),
    re.compile(r'Chain of Thought:.*?(?=\n[A-Z]|$)', re.DOTALL),
    re.compile(r'Step-by-step reasoning:.*?(?=\n[A-Z]|$)', re.DOTALL | re.IGNORECASE),
    re.compile(r'Let me think.*?(?=\n[A-Z]|$)', re.DOTALL | re.IGNORECASE),
]

MEDICAL_DISCLAIMER = (
    "\n\n---\n"
    "*This information is for educational purposes only and does not replace professional "
    "medical advice. Please consult a qualified healthcare provider for diagnosis and treatment.*"
)


def clean_response(raw_output: str, add_disclaimer: bool = True) -> str:
    """
    Remove all reasoning/thinking tokens from model output.
    Returns clean, professional doctor response.
    """
    text = raw_output

    # Strip all think-pattern variants
    for pattern in _THINK_PATTERNS:
        text = pattern.sub('', text)

    # Remove common reasoning section headers left over
    for header in [
        'Expert Medical Answer:', 'Medical Answer:', 'Answer:', 'Doctor:',
        'Response:', 'Final Answer:',
    ]:
        if text.lstrip().startswith(header):
            text = text.lstrip()[len(header):]

    # Clean up extra whitespace / blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()

    # Safety fallback: if cleaning removed everything, provide safe default
    if not text:
        text = (
            'I was unable to generate a clear response. '
            'Please consult a qualified healthcare provider for this concern.'
        )

    if add_disclaimer:
        text += MEDICAL_DISCLAIMER

    return text


print('✅ clean_response() defined — all think/reasoning tokens will be stripped')

In [ ]:
# ─── 9. Prompt Builder with Chat Memory ────────────────────────────────────

SYSTEM_PROMPT = (
    'You are a board-certified medical doctor. '
    'Provide clear, accurate, and compassionate medical information. '
    'Be concise, professional, and structured. '
    'Never expose internal reasoning. Respond directly as a doctor would.'
)


def build_prompt(
    current_question: str,
    previous_summary: Optional[str] = None,
    conversation_history: Optional[List[dict]] = None,
    max_history_turns: int = 3,
) -> str:
    """
    Build the full prompt with optional chat memory injection.

    Args:
        current_question   : The patient's current message.
        previous_summary   : Summarized history from prior sessions.
        conversation_history: List of {role, content} dicts (recent turns).
        max_history_turns  : How many recent turns to include.

    Returns:
        Tokenizer-formatted chat string.
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Inject previous session summary as assistant context
    if previous_summary and previous_summary.strip():
        history_context = (
            f"Patient History:\n{previous_summary.strip()}\n\n"
            f"Current Question:\n{current_question.strip()}\n\n"
            "Provide a medical response."
        )
        messages.append({"role": "user", "content": history_context})
    else:
        # Include recent conversation turns (in-session memory)
        if conversation_history:
            recent = conversation_history[-max_history_turns * 2:]  # user+assistant pairs
            for turn in recent:
                messages.append({"role": turn["role"], "content": turn["content"]})
        messages.append({"role": "user", "content": current_question.strip()})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


print('✅ build_prompt() with chat memory injection defined')

In [ ]:
# ─── 10. Core Inference Function ───────────────────────────────────────────

@torch.inference_mode()
def generate_medical_response(
    current_question: str,
    previous_summary: Optional[str] = None,
    conversation_history: Optional[List[dict]] = None,
    max_new_tokens: int = None,
    temperature: float = None,
) -> str:
    """
    Generate a clean medical response with reasoning stripped.

    Returns:
        Clean doctor response string (no think tags, with disclaimer).
    """
    max_new_tokens = max_new_tokens or INFERENCE_CFG['max_new_tokens']
    temperature    = temperature    or INFERENCE_CFG['temperature']

    prompt = build_prompt(
        current_question=current_question,
        previous_summary=previous_summary,
        conversation_history=conversation_history,
    )

    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=2048,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=INFERENCE_CFG['top_p'],
            repetition_penalty=INFERENCE_CFG['repetition_penalty'],
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (not the prompt)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return clean_response(raw_output)


print('✅ generate_medical_response() defined')

In [ ]:
# ─── 11. Smoke Test ────────────────────────────────────────────────────────
test_response = generate_medical_response(
    current_question='What are the common symptoms of type 2 diabetes?',
    temperature=0.6,
    max_new_tokens=200,
)
print('--- Smoke Test Response ---')
print(test_response)
print('---')
assert '<think' not in test_response.lower(), 'ERROR: think tag leaked!'
assert '</think>' not in test_response.lower(), 'ERROR: think closing tag leaked!'
print('✅ Smoke test passed — no reasoning tokens in output')

In [ ]:
# ─── 12. FastAPI Application ───────────────────────────────────────────────

app = FastAPI(
    title='Medical AI Inference API',
    description='Llama 3.1-8B + LoRA | Clean medical responses | No reasoning exposure',
    version='1.0.0',
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)


# ── Request / Response Models ──────────────────────────────────────────────

class ChatRequest(BaseModel):
    message: str
    previous_summary: Optional[str] = None
    context: Optional[List[dict]] = None    # [{"role":"user","content":"..."}, ...]
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.6
    mode: Optional[str] = 'medical'        # 'medical' | 'symptom_checker'


class ChatResponse(BaseModel):
    reply: str
    model: str = 'llama-3.1-8b-medical-lora'


# ── Endpoints ─────────────────────────────────────────────────────────────

@app.get('/health')
def health():
    return {'status': 'healthy', 'model': 'llama-3.1-8b-medical-lora'}


@app.post('/api/v1/chat/message', response_model=ChatResponse)
def chat_message(request: ChatRequest):
    """
    Primary chat endpoint — compatible with existing backend.

    JSON input:
    {
        "message": "patient question",
        "previous_summary": "optional summary from prior session",
        "context": [{"role": "user", "content": "..."}],
        "max_tokens": 512,
        "temperature": 0.6
    }

    JSON output:
    {
        "reply": "clean doctor response",
        "model": "llama-3.1-8b-medical-lora"
    }
    """
    try:
        reply = generate_medical_response(
            current_question=request.message,
            previous_summary=request.previous_summary,
            conversation_history=request.context,
            max_new_tokens=request.max_tokens,
            temperature=request.temperature,
        )
        return ChatResponse(reply=reply)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print('✅ FastAPI app defined')

In [ ]:
# ─── 13. Start Server + Expose via Ngrok ──────────────────────────────────
if INFERENCE_CFG['use_ngrok']:
    from pyngrok import ngrok
    # Set your ngrok auth token: ngrok.set_auth_token('YOUR_NGROK_TOKEN')
    public_url = ngrok.connect(INFERENCE_CFG['port'])
    print(f'\n🌐 Public URL (use this as CUSTOM_API_URL in your backend):')
    print(f'   {public_url}')
    print(f'   Endpoint: {public_url}/api/v1/chat/message')
else:
    print(f'Server running at http://{INFERENCE_CFG["host"]}:{INFERENCE_CFG["port"]}')
    print(f'Endpoint: http://{INFERENCE_CFG["host"]}:{INFERENCE_CFG["port"]}/api/v1/chat/message')

uvicorn.run(
    app,
    host=INFERENCE_CFG['host'],
    port=INFERENCE_CFG['port'],
    log_level='info',
)